# 05 - NLP com embeddings semanticos locais (Hugging Face)

Este notebook usa um modelo Sentence-BERT local do Hugging Face para transformar cada texto em um vetor semantico denso. A abordagem TF-IDF fica como baseline lexical, enquanto a proposta principal usa embeddings do modelo `sentence-transformers/all-MiniLM-L6-v2`.

Nao ha SMOTE ou geracao sintetica de textos. O tratamento de diferencas entre classes e feito por pesos no classificador, e a avaliacao usa precision, recall e F1.

In [ ]:
from pathlib import Path
import os
import sys

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.model import train_nlp_xgb, train_semantic_classifier
from src.preprocessing import (
    DEFAULT_SENTENCE_TRANSFORMER_MODEL,
    get_embedding_cache_path,
    get_semantic_embeddings,
    get_tfidf_vectorizer,
    process_text_data,
)

In [ ]:
def metricas_classificacao(y_true, y_pred, target_names):
    report = classification_report(
        y_true,
        y_pred,
        target_names=target_names,
        output_dict=True,
        zero_division=0
    )
    tabela = pd.DataFrame(report).transpose()
    tabela = tabela.drop(index='accuracy', errors='ignore')
    tabela = tabela[['precision', 'recall', 'f1-score', 'support']]
    tabela['support'] = tabela['support'].astype(int)
    return tabela

## Carga dos dados

O mesmo split estratificado e usado nos dois experimentos para que a comparacao seja justa.

In [ ]:
texts, y, le_nlp = process_text_data('data/Mental Health Disorder Detection Dataset.csv')
target_names = le_nlp.classes_

X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    texts,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Distribuicao das categorias:')
display(pd.Series(le_nlp.inverse_transform(y)).value_counts().sort_index())

## Baseline lexical: TF-IDF + XGBoost

Este baseline representa textos por frequencia de termos e n-gramas. Ele e mantido apenas como ponto de comparacao com o modelo semantico.

In [ ]:
vectorizer = get_tfidf_vectorizer()
X_train_vec = vectorizer.fit_transform(X_train_t)
X_test_vec = vectorizer.transform(X_test_t)

modelo_tfidf = train_nlp_xgb(X_train_vec, y_train_t, X_test_vec, y_test_t)
pred_tfidf = modelo_tfidf.predict(X_test_vec)
prob_tfidf = modelo_tfidf.predict_proba(X_test_vec)

relatorio_tfidf = metricas_classificacao(y_test_t, pred_tfidf, target_names)
display(relatorio_tfidf)

## Representacao semantica: Sentence-BERT local

O modelo abaixo e baixado uma vez do Hugging Face e depois fica no cache local. Os embeddings tambem sao salvos em `exports/embeddings`, evitando recalculo em novas execucoes com o mesmo split.

In [ ]:
embedding_model_name = DEFAULT_SENTENCE_TRANSFORMER_MODEL
print(f'Modelo de embeddings: {embedding_model_name}')

train_cache = get_embedding_cache_path(X_train_t, 'train', embedding_model_name)
test_cache = get_embedding_cache_path(X_test_t, 'test', embedding_model_name)

X_train_emb = get_semantic_embeddings(
    X_train_t,
    model_name=embedding_model_name,
    cache_path=train_cache,
    batch_size=32
)
X_test_emb = get_semantic_embeddings(
    X_test_t,
    model_name=embedding_model_name,
    cache_path=test_cache,
    batch_size=32
)

print(f'Formato dos embeddings de treino: {X_train_emb.shape}')
print(f'Formato dos embeddings de teste: {X_test_emb.shape}')

In [ ]:
modelo_semantico = train_semantic_classifier(X_train_emb, y_train_t)
pred_semantico = modelo_semantico.predict(X_test_emb)
prob_semantico = modelo_semantico.predict_proba(X_test_emb)

relatorio_semantico = metricas_classificacao(y_test_t, pred_semantico, target_names)
display(relatorio_semantico)

## Comparacao por F1-macro

O F1-macro da o mesmo peso para cada classe, por isso e mais adequado do que usar uma medida global que pode esconder desempenho ruim em classes especificas.

In [ ]:
comparacao = pd.DataFrame({
    'precision_macro': [
        relatorio_tfidf.loc['macro avg', 'precision'],
        relatorio_semantico.loc['macro avg', 'precision']
    ],
    'recall_macro': [
        relatorio_tfidf.loc['macro avg', 'recall'],
        relatorio_semantico.loc['macro avg', 'recall']
    ],
    'f1_macro': [
        relatorio_tfidf.loc['macro avg', 'f1-score'],
        relatorio_semantico.loc['macro avg', 'f1-score']
    ]
}, index=['TF-IDF + XGBoost', 'Sentence-BERT + Logistic Regression'])

display(comparacao.sort_values('f1_macro', ascending=False))

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(
    confusion_matrix(y_test_t, pred_semantico),
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=target_names,
    yticklabels=target_names
)
plt.title('Matriz de confusao - Sentence-BERT')
plt.xlabel('Predito')
plt.ylabel('Real')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
os.makedirs('exports/models', exist_ok=True)
joblib.dump(modelo_semantico, 'exports/models/nlp_semantic_classifier.pkl')
joblib.dump(
    {
        'embedding_model_name': embedding_model_name,
        'normalize_embeddings': True,
        'train_cache': train_cache,
        'test_cache': test_cache
    },
    'exports/models/nlp_semantic_config.pkl'
)
joblib.dump(le_nlp, 'exports/models/nlp_encoder.pkl')
joblib.dump(modelo_tfidf, 'exports/models/nlp_tfidf_xgb_model.pkl')
joblib.dump(vectorizer, 'exports/models/nlp_tfidf_vectorizer.pkl')
print('Modelos e configuracoes salvos em exports/models.')

## Teste com texto novo

A funcao abaixo usa o mesmo modelo semantico local para transformar um novo texto e gerar a predicao.

In [ ]:
def prever_texto_semantico(texto):
    emb = get_semantic_embeddings(
        [texto],
        model_name=embedding_model_name,
        batch_size=1,
        show_progress_bar=False
    )
    pred = modelo_semantico.predict(emb)[0]
    prob = modelo_semantico.predict_proba(emb)[0].max()
    return le_nlp.inverse_transform([pred])[0], prob

texto_teste = 'I feel anxious every day and my heart races when I need to leave home.'
classe, confianca = prever_texto_semantico(texto_teste)
print(f'Texto: {texto_teste}')
print(f'Predicao: {classe} ({confianca:.2%})')